# Connect to Mongodb

In [ ]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from pprint import pprint
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("MONGO_URI"))

In [ ]:
client = MongoClient(os.getenv("MONGO_URI"))

In [ ]:
try:
    client.admin.command("ping")
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
list_databases = client.list_database_names()
pprint(list_databases)

In [ ]:
db = client["mongodb_book"]
pprint(db.list_collection_names())

In [ ]:
collection = db["book"]
vs_ = db["chunked_docs"]
full_collection = db["full_docs"]

#collection.insert_one({"title": "MongoDB Basics", "author": "John Doe", "year": 2021})

# Vector Search Index

## define

In [ ]:
definition = {
    "fields":[
        {
            "type": "vector",
            "path": "embedding",
            "numDimensions": 1024,
            "similarity": "cosine"
        }
    ]
}

In [ ]:
from pymongo.errors import OperationFailure

Search_Index_Model = SearchIndexModel(
    definition=definition,
    name="vector_search_index",
    type="vectorSearch"
)

try:
    name = collection.create_search_index(Search_Index_Model)
    print(f"Created search index {name!r}")
except OperationFailure as e:
    # the index survives between lessons, so re-running this cell is expected to clash
    print(f"Could not create the index, continuing with the existing one: {e}")

## get information form a PDF file

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader("data/mongodb_handbook.pdf")

pages = loader.load()

print(f"Total Pages: {len(pages)}")

cleaned_pages = []
for page in pages:
    if len(page.page_content.split(" ")) > 20:
        cleaned_pages.append(page)

print(f"Total Cleaned Pages: {len(cleaned_pages)}")

## Create chuncks from documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 750,
    chunk_overlap = 150
)
docs_chunks = text_splitter.split_documents(cleaned_pages)

In [ ]:
docs_chunks[:10]
print(docs_chunks[0])

In [ ]:
print(f"Total Chunks: {len(docs_chunks)}")

# RAG - Retrieval-augmented generation

## creating the embeddings

In [ ]:
import voyageai

# named `voyage`, not `client`: `client` is the MongoClient from the top of the notebook
voyage = voyageai.Client(api_key=os.getenv("VOYAGEAI_API_KEY"))

EMBED_MODEL = "voyage-4"

In [ ]:
# Every document inserted by the earlier lessons carries the embedding of "test text",
# so they all sit at the same point in the vector space. Clear them before re-embedding.
print(collection.count_documents({}), "documents before reset")
collection.delete_many({})
print(collection.count_documents({}), "documents after reset")

In [ ]:
BATCH_SIZE = 64
records = []

for start in range(0, len(docs_chunks), BATCH_SIZE):
    batch = docs_chunks[start:start + BATCH_SIZE]
    texts = [chunk.page_content for chunk in batch]

    # one embedding per chunk, from that chunk's own text
    embeddings = voyage.embed(
        texts,
        model=EMBED_MODEL,
        input_type="document"
    ).embeddings

    for chunk, embedding in zip(batch, embeddings):
        records.append({
            "text": chunk.page_content,
            "metadata": chunk.metadata,
            "embedding": embedding,
        })

    print(f"Embedded {len(records)} / {len(docs_chunks)} chunks")

collection.insert_many(records)
print(f"Inserted {len(records)} documents, {len(records[0]['embedding'])} dimensions each")

## Creating the query

In [ ]:
query_text = "how to do aggregate pipeline in mongodb"
query_vector = voyage.embed(
    [query_text],
    model=EMBED_MODEL,
    input_type="query"
).embeddings[0]

print(len(query_vector))

In [ ]:
pipeline = [
    {
        "$vectorSearch": {
            "exact": True,
            "index": "vector_search_index",
            "limit": 3,
            "path": "embedding",
            "queryVector": query_vector,
        }
    },
    {
        "$project": {
            "_id": 0,
            "text": 1,
            "page": "$metadata.page",
            "score": {
                "$meta": "vectorSearchScore"
            }
        }
    }
]

In [ ]:
response = collection.aggregate(pipeline)


In [ ]:
context_string = ""
for result in response:
    print(f"page {result['page']}  score {result['score']:.4f}")
    context_string = context_string + " " + result["text"]

print(context_string)

# 

# Use Chat Google AI

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.1,
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [ ]:
#define rag promp
prompt = ChatPromptTemplate.from_messages([
    ("system","Answer the quetion based ONLY on the following context:\n\n {context_string}"),
    ("human", "{query}")

])
print(prompt)

In [ ]:
'''prompt = f"""Use the following pieces of context to answer the question at the end.
    {context_string}
    Question: {query_text}"""'''

In [ ]:
rag_chain = prompt | llm | StrOutputParser()

output = rag_chain.invoke({
        "context_string": context_string,
        "query": query_text
})

In [ ]:
print(output)